In [9]:
#Cell1： Install and upgrade required dependencies

!pip install transformers --upgrade -q
!pip install peft accelerate -q
!pip install torchao --upgrade -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.5 MB/s eta 0:00:00


In [2]:
#Cell2： Mount Google Drive and extract competition data

from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

ZIP_PATH = "/content/drive/MyDrive/pixels-to-predictions.zip"
BASE_DIR = "/content/pixels-to-predictions"

if not os.path.exists(BASE_DIR):
    print("Extracting data...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(BASE_DIR)
    print("Extraction complete")
else:
    print("Data already exists, skipping extraction")

Mounted at /content/drive
Extracting data...
Extraction complete


In [3]:
#Cell3： Imports and configuration — defines all hyperparameters, paths, and global settings

import json, ast, random, os
import numpy as np
import pandas as pd
from PIL import Image
import torchvision.transforms as T
import torch
from torch.utils.data import Dataset, DataLoader

try:
    from transformers import AutoModelForVision2Seq
except ImportError:
    from transformers import AutoModelForImageTextToText as AutoModelForVision2Seq

from transformers import AutoProcessor, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from tqdm.notebook import tqdm

import transformers
print(f"transformers: {transformers.__version__}")
print(f"numpy: {np.__version__}")
print("✓ Imports successful")

class CFG:
    model_name   = "HuggingFaceTB/SmolVLM-500M-Instruct"
    base_dir     = "/content/pixels-to-predictions"
    train_csv    = f"{base_dir}/train.csv"
    val_csv      = f"{base_dir}/val.csv"
    test_csv     = f"{base_dir}/test.csv"
    sample_sub   = f"{base_dir}/sample_submission.csv"
    image_dir    = f"{base_dir}/images/images"
    output_dir   = "/content/drive/MyDrive/smolvlm_output"
    caption_cache_path = f"{output_dir}/caption_cache.json"

    seed         = 42
    image_size   = 384
    max_length   = 640
    epochs       = 4
    batch_size   = 2
    grad_accum   = 8
    lr           = 2e-4
    weight_decay = 0.01
    warmup_ratio = 0.10

    lora_r       = 7
    lora_alpha   = 14
    lora_dropout = 0.05
    use_dora     = True
    lora_target  = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ]

    use_val_in_train       = True
    use_image_caption      = True
    use_metadata           = True
    grad_checkpointing     = True
    caption_max_new_tokens = 60

os.makedirs(CFG.output_dir, exist_ok=True)

def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_everything(CFG.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")


transformers: 5.8.0
numpy: 2.0.2
✓ Imports successful
Device: cuda
GPU: Tesla T4


In [4]:
#Cell4：Load train/val/test CSVs and verify image paths

def parse_choices(s):
    try:    return json.loads(s)
    except Exception:
        try: return ast.literal_eval(s)
        except Exception: return str(s).split(",")

def load_split(path):
    df = pd.read_csv(path)
    df["choices_list"] = df["choices"].apply(parse_choices)
    return df

train_df = load_split(CFG.train_csv)
val_df   = load_split(CFG.val_csv)
test_df  = load_split(CFG.test_csv)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

def get_image_path(row):
    rel = row["image_path"]
    rel_strip = rel.replace("images/", "", 1) if rel.startswith("images/") else rel
    return os.path.join(CFG.image_dir, rel_strip)

ok = sum(os.path.exists(get_image_path(r)) for _, r in train_df.head(10).iterrows())
print(f"Path 1 (images/images/): {ok}/10")

if ok == 0:
    def get_image_path(row):
        return os.path.join(CFG.base_dir, row["image_path"])
    ok2 = sum(os.path.exists(get_image_path(r)) for _, r in train_df.head(10).iterrows())
    print(f"Path 2 (base_dir/): {ok2}/10")
    if ok2 == 0:
        print("Directory structure:")
        for root, dirs, files in os.walk(CFG.base_dir):
            level = root.replace(CFG.base_dir, '').count(os.sep)
            if level < 3:
                print('  ' * level + os.path.basename(root) + '/')


Train: 3109 | Val: 1048 | Test: 1008
Path 1 (images/images/): 10/10


In [5]:
#Cell5 Load the SmolVLM processor for text and image preprocessing

print("Loading processor...")
processor = AutoProcessor.from_pretrained(
    CFG.model_name,
    trust_remote_code=True,
)
LETTER_MAP = ["A", "B", "C", "D", "E"]
print("✓ Processor loaded successfully")

Loading processor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✓ Processor loaded successfully


In [6]:
#Cell6： Generate image captions using BLIP — results are cached to Google Drive to avoid recomputation

from transformers import BlipProcessor, BlipForConditionalGeneration

@torch.no_grad()
def generate_captions_blip(df_list, cache_path):
    if os.path.exists(cache_path):
        print(f"Loading captions from cache: {cache_path}")
        with open(cache_path) as f:
            return json.load(f)

    print("Generating captions with BLIP (approximately 15-20 minutes)...")
    blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    blip_model     = BlipForConditionalGeneration.from_pretrained(
        "Salesforce/blip-image-captioning-base",
        torch_dtype=torch.float16,
    ).to(device)
    blip_model.eval()

    all_rows = pd.concat(df_list, ignore_index=True).drop_duplicates("id")
    cache = {}

    for _, row in tqdm(all_rows.iterrows(), total=len(all_rows), desc="Captioning"):
        try:
            img = Image.open(get_image_path(row)).convert("RGB")
        except Exception:
            cache[row["id"]] = ""
            continue

        inputs = blip_processor(images=img, return_tensors="pt").to(device, torch.float16)
        out    = blip_model.generate(**inputs, max_new_tokens=60)
        caption = blip_processor.decode(out[0], skip_special_tokens=True).strip()
        cache[row["id"]] = caption

    del blip_model, blip_processor
    torch.cuda.empty_cache()

    with open(cache_path, "w") as f:
        json.dump(cache, f, ensure_ascii=False)
    print(f"✓ Caption cache saved ({len(cache)} entries) → {cache_path}")
    return cache

caption_cache = generate_captions_blip(
    [train_df, val_df, test_df],
    CFG.caption_cache_path,
)

# Print a few sample captions
for rid, cap in list(caption_cache.items())[:3]:
    print(f"  [{rid}] {cap}")

Loading captions from cache: /content/drive/MyDrive/smolvlm_output/caption_cache.json
  [train_07667] a frog and a frog are sitting on a leaf
  [train_02628] a couple of monkeys
  [train_00927] a group of lions


In [7]:
#Cell7: Prompt builder — constructs the full input prompt from image caption, metadata, lecture, hint, question, and choices

def build_prompt(row, caption_cache=None):
    parts = []

    if CFG.use_image_caption and caption_cache:
        cap = caption_cache.get(row["id"], "").strip()
        if cap:
            parts.append(f"Image description: {cap}")

    if CFG.use_metadata:
        meta = " | ".join(
            str(row.get(f, "") or "").strip()
            for f in ["subject", "grade", "topic", "category", "skill"]
            if str(row.get(f, "") or "").strip()
        )
        if meta:
            parts.append(f"Context: {meta}")

    lec = str(row.get("lecture", "") or "").strip()
    if lec:
        parts.append(f"Background: {lec}")

    hint = str(row.get("hint", "") or "").strip()
    if hint:
        parts.append(f"Hint: {hint}")

    parts.append(f"Question: {str(row['question']).strip()}")

    choices = row["choices_list"]
    choice_lines = "\n".join(f"  {LETTER_MAP[i]}) {c}" for i, c in enumerate(choices))
    parts.append(f"Choices:\n{choice_lines}")
    parts.append("The correct answer is:")
    return "\n\n".join(parts)

print(build_prompt(train_df.iloc[0], caption_cache)[:300])
print("✓ Prompt builder working correctly")

Image description: a frog and a frog are sitting on a leaf

Context: natural science | grade8 | literacy-in-science | Adaptations and natural selection | How can animal behaviors affect reproductive success? Identify evidence to support a claim

Background: Animals increase their reproductive succes
✓ Prompt builder working correctly


In [8]:
#Cell8: Dataset, transforms, and DataLoader setup — includes image augmentation for training and collate function

def get_transform(is_train):
    if is_train:
        return T.Compose([
            T.RandomResizedCrop(CFG.image_size, scale=(0.85, 1.0)),
            T.RandomHorizontalFlip(p=0.3),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            T.ToTensor(), T.ToPILImage(),
        ])
    return T.Compose([
        T.Resize((CFG.image_size, CFG.image_size)),
        T.ToTensor(), T.ToPILImage(),
    ])

class SciVQADataset(Dataset):
    def __init__(self, df, caption_cache=None, is_train=True):
        self.df            = df.reset_index(drop=True)
        self.caption_cache = caption_cache or {}
        self.is_train      = is_train
        self.transform     = get_transform(is_train)

    def __len__(self): return len(self.df)

    def _load_image(self, row):
        try:
            img = Image.open(get_image_path(row)).convert("RGB")
        except Exception:
            img = Image.new("RGB", (CFG.image_size, CFG.image_size), 128)
        return self.transform(img)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        image  = self._load_image(row)
        prompt = build_prompt(row, self.caption_cache)
        messages = [{"role": "user", "content": [
            {"type": "image"}, {"type": "text", "text": prompt}
        ]}]
        answer = int(row["answer"]) if self.is_train and "answer" in row else None
        target = LETTER_MAP[answer] if answer is not None else ""
        return {
            "messages": messages, "image": image,
            "target": target, "id": row["id"],
            "num_choices": int(row["num_choices"]),
        }

def make_collate(is_train=True):
    def collate(batch):
        texts = [processor.apply_chat_template(b["messages"], add_generation_prompt=True)
                 for b in batch]
        enc = processor(
            text=texts,
            images=[b["image"] for b in batch],
            return_tensors="pt",
            padding=True,
            truncation=False,   # disable truncation to avoid image token mismatch
        )
        if is_train:
            labels = enc["input_ids"].clone().fill_(-100)
            for i, b in enumerate(batch):
                tgt = b["target"]
                if not tgt: continue
                tgt_id  = processor.tokenizer(tgt, add_special_tokens=False)["input_ids"]
                seq_len = (enc["input_ids"][i] != processor.tokenizer.pad_token_id).sum()
                if seq_len > 0 and tgt_id:
                    labels[i, seq_len - 1] = tgt_id[0]
            enc["labels"] = labels
        enc["ids"]         = [b["id"] for b in batch]
        enc["num_choices"] = [b["num_choices"] for b in batch]
        return enc
    return collate

train_src = pd.concat([train_df, val_df], ignore_index=True) if CFG.use_val_in_train else train_df

train_ds = SciVQADataset(train_src, caption_cache, is_train=True)
val_ds   = SciVQADataset(val_df,    caption_cache, is_train=True)
test_ds  = SciVQADataset(test_df,   caption_cache, is_train=False)

kw = dict(num_workers=2, pin_memory=True)
train_loader = DataLoader(train_ds, CFG.batch_size, shuffle=True,  collate_fn=make_collate(True),  **kw)
val_loader   = DataLoader(val_ds,   CFG.batch_size*2, shuffle=False, collate_fn=make_collate(True),  **kw)
test_loader  = DataLoader(test_ds,  CFG.batch_size*2, shuffle=False, collate_fn=make_collate(False), **kw)

print(f"✓ Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

✓ Train: 2079 | Val: 262 | Test: 252


In [11]:
#Cell9 Load SmolVLM-500M and apply LoRA (with DoRA) to attention and MLP layers

print("Loading model and applying DoRA...")
base_model = AutoModelForVision2Seq.from_pretrained(
    CFG.model_name,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
if CFG.grad_checkpointing:
    base_model.gradient_checkpointing_enable()

lora_cfg = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = CFG.lora_r,
    lora_alpha     = CFG.lora_alpha,
    lora_dropout   = CFG.lora_dropout,
    target_modules = CFG.lora_target,
    use_dora       = CFG.use_dora,
    bias           = "none",
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()
model = model.to(device)
print("✓ Model ready")


Loading model and applying DoRA...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

trainable params: 4,490,240 || all params: 511,972,544 || trainable%: 0.8770
✓ Model ready


In [13]:
#Cell10: Training loop with TTA inference — trains the model and evaluates using logit-based scoring with 3 prompt variants

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CFG.lr, weight_decay=CFG.weight_decay,
)
total_steps  = (len(train_loader) // CFG.grad_accum) * CFG.epochs
warmup_steps = int(total_steps * CFG.warmup_ratio)
scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

def train_epoch(model, loader, optimizer, scheduler, epoch):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(loader, desc=f"Epoch {epoch}")):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        pv   = batch.get("pixel_values")
        if pv is not None: pv = pv.to(device)
        lbl  = batch["labels"].to(device)
        kw   = dict(input_ids=ids, attention_mask=mask, labels=lbl)
        if pv is not None: kw["pixel_values"] = pv
        loss = model(**kw).loss / CFG.grad_accum
        loss.backward()
        total_loss += loss.item() * CFG.grad_accum
        if (step + 1) % CFG.grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
    return total_loss / len(loader)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    letter_ids = [
        processor.tokenizer(l, add_special_tokens=False)["input_ids"][0]
        for l in LETTER_MAP
    ]
    # TTA: average logit scores across 3 different prompt endings
    tta_suffixes = [
        "The correct answer is:",
        "Answer:",
        "Therefore, the answer is:",
    ]
    all_ids, all_preds = [], []
    for batch in tqdm(loader, desc="Inference"):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        pv   = batch.get("pixel_values")
        if pv is not None: pv = pv.to(device)
        batch_size  = ids.size(0)
        vote_scores = [[] for _ in range(batch_size)]
        for suffix in tta_suffixes:
            kw = dict(input_ids=ids, attention_mask=mask)
            if pv is not None: kw["pixel_values"] = pv
            logits = model(**kw).logits
            for i in range(batch_size):
                pos    = mask[i].sum().item() - 1
                nc     = batch["num_choices"][i]
                scores = [logits[i, pos, letter_ids[j]].item() for j in range(nc)]
                vote_scores[i].append(scores)
        for i in range(batch_size):
            nc = batch["num_choices"][i]
            avg_scores = [
                sum(vote_scores[i][t][j] for t in range(len(tta_suffixes))) / len(tta_suffixes)
                for j in range(nc)
            ]
            all_ids.append(batch["ids"][i])
            all_preds.append(int(np.argmax(avg_scores)))
    return all_ids, all_preds

def evaluate(model, val_df, loader):
    ids, preds = predict(model, loader)
    pred_map   = dict(zip(ids, preds))
    correct    = sum(
        pred_map.get(row["id"], -1) == int(row["answer"])
        for _, row in val_df.iterrows()
    )
    return correct / len(val_df)

best_acc  = 0.0
best_path = os.path.join(CFG.output_dir, "best_model_v2")

for epoch in range(1, CFG.epochs + 1):
    loss = train_epoch(model, train_loader, optimizer, scheduler, epoch)
    acc  = evaluate(model, val_df, val_loader)
    print(f"Epoch {epoch} | loss={loss:.4f} | val_acc={acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        model.save_pretrained(best_path)
        processor.save_pretrained(best_path)
        print(f"  ✓ val_acc={acc:.4f}")

print(f"\nBest Val Accuracy: {best_acc:.4f}")

Epoch 1:   0%|          | 0/2079 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
#Cell11: Inference Code; Load best saved model and generate submission.csv for Kaggle

print("Loading best model...")
base2       = AutoModelForVision2Seq.from_pretrained(
    CFG.model_name,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
infer_model = PeftModel.from_pretrained(base2, best_path).to(device)

test_ids, test_preds = predict(infer_model, test_loader)

sub    = pd.DataFrame({"id": test_ids, "answer": test_preds})
sample = pd.read_csv(CFG.sample_sub)
sub    = sample[["id"]].merge(sub, on="id", how="left")
sub["answer"] = sub["answer"].fillna(0).astype(int)

sub.to_csv(os.path.join(CFG.output_dir, "submission.csv"), index=False)
sub.to_csv("/content/submission.csv", index=False)

print("✓ Done!")
print(f"Answer distribution:\n{sub['answer'].value_counts().sort_index().to_string()}")
print(sub.head(10).to_string())


加载最佳模型...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Inference:   0%|          | 0/252 [00:00<?, ?it/s]

✓ 完成！
答案分布:
answer
0    320
1    350
2    253
3     72
4     13
           id  answer
0  test_01750       1
1  test_00128       0
2  test_02891       3
3  test_02425       4
4  test_00930       2
5  test_03725       2
6  test_00009       1
7  test_02880       1
8  test_01208       1
9  test_00619       0


In [ ]:
#Cell12： File Download

from google.colab import files
files.download("/content/submission.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>